# Performance.py

This notebook is used to calculate the performance of a `sentence-transformers` model on a predefined set of validation examples. The validation set consists of 60 thousand enhanced and scrambled LOINC codes (enhancement means we have applied TtC synonym resolution, related-name insertion, word order permutation, and character deletion), 20 thousand each of long common name, short name, and display name. The file lives as a Data Asset in Azure Blob Storage and is read directly into local memory for computation.

Model performance is measured in several dimensions, each broken down by the number of neighbors `K` retrieved by the ANN search:

* Top-K accuracy: the percentage of the time that the correct standardized code is in the K highest scoring search results
* Mean rank: the average position (1st, 2nd, 3rd, etc.) in the list of returned neighbors (sorted by score) of the correct standardized code, when present
* Mean high cosine similarity: the average cosine similarity between the nonstandard input and the **highest** scoring search result--this result is not guaranteed to be the correct answer
* Mean right cosine similarity: the average cosine similarity between the nonstandard input and the **correct** standard code, if it appears in the top-K search results (for a particular search, if the right answer isn't found, than that search doesn't contribute to the mean calculation; only searches in which the right answer is present are used)
* Mean search time: the time it takes to retrieve the list of neighbors

Additionally, the encoding time for the model (the time it takes the model to transform an input free-text string into a vector of the embedding dimension) is computed and reported once (i.e. not stratified by K-value), since this time doesn't change as K increases.

While it is possible to run this notebook on CPU, we highly recommend using an appropriately powerful GPU (e.g. the NC24 A100 series) to speed up computation. The Approximate Nearest Neighbor search we use makes searching in memory almost instantaneous, but there is still a time-cost to encode each validation string input into a vector before semantic searching. Over 60 thousand encodings, this time adds up: GPU encoding is 10-15 times faster than CPU.


## Setup

Make sure that once the compute instance is running, you activate the kernel associated with the DIBBs Env in the upper right dropdown. Its packages are correctly optimized for this notebook and avoids some `numpy` instabilities plaguing Azure.

In [1]:
pip install azure-keyvault-secrets azure-identity azure-ai-ml azureml-fsspec hnswlib

Note: you may need to restart the kernel to use updated packages.


Now we'll do our basic, standard authentication work. We need all these variables to be able to access our container storage from a file mount. The `DATASTORE_NAME` is not a protected secret and therefore doesn't need to be stashed in KeyVault, as it's a standard Azure default.

In [2]:
# Authenticate to Key Vault
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

credential = DefaultAzureCredential()
key_vault = "dibbsttc6059789213"
secret_client = SecretClient(vault_url=f"https://{key_vault}.vault.azure.net/", credential=credential)

SUBSCRIPTION = secret_client.get_secret("subscription").value
RESOURCE_GROUP = secret_client.get_secret("resource-group").value
WS_NAME = secret_client.get_secret("workspace-name").value
DATASTORE_NAME = 'workspaceblobstore'

_IMPORTANT_: This step can't be skipped, even though we're not directly using any of the `ml_client` functionality. This authentication and connection step allows us to use this notebook cleanly within our compute ecosystem. Basically, instantiating the class object acts as a connection that allows us to do everything that follows.

In [3]:
from azure.ai.ml import MLClient

# Authenticate and connect to workspace
ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION,
    RESOURCE_GROUP,
    WS_NAME,
)

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Finally, we'll set the remainder of our imports and some global constants we'll be using.

The most important variables in this cell are the `MODEL VARIABLES` values, the `MODLE_NAME` and `EMBEDDING_SIZE`. The model name comes directly from the hugging face page for a particular model, and can be directly copied using the button beside the name on the web page. The embedding size for most models we work with is 768 (which is the industry standard dimensionality for any BERT-based transformer), but some are 1024 and a few are even higher. You can refer to the spreadsheet that tracks model performance for the exact size of a given model. If you set an embedding size here that proves to be incorrect later, when calculating the index, no harm will be done--the cell will simply halt and tell you the embedding dimensionality is wrong. Just come back here and change the value (likely to whichever of 1024 or 768 you haven't tried yet), then go back and pick up with indexing.

In [4]:
import os
import random
import time
from typing import List

# MODEL VARIABLES
MODEL_NAME = "intfloat/e5-large-v2"
EMBEDDING_SIZE = 1024

# The name of the file in blob storage of the model's pickled vectors
SNOINC_CODES_FILE = "./loinc_lab_names_20251008.csv"
DATE = SNOINC_CODES_FILE.split("_")[-1].split(".")[0]
MODEL_FOLDER_NAME = f"loinc_lab_names_{MODEL_NAME.replace('/', '_')}_{DATE}"
MODEL_FOLDER = f"embeddings/refined/split/{MODEL_FOLDER_NAME}"

# The name of the HNSW index file for this particular model
INDEX_FP = f"hnswlib_index_{MODEL_NAME.replace('/', '_')}_400.index"

# VALIDATION VARIABLES
VALIDATION_FILE = "./validation_set_60k_pairs.txt"
K_VALUES = [1, 3, 5, 10]

**Important**: This cell determines whether the notebook will use Exact Neareset Neighbor search or Approximate Nearest Neighbor Search. Exact search is supported _only_ when the notebook is run using a GPU-enabled cluster, since the Tensor operations are valid only in a CUDA environment with `pytorch`. Further, exact search is computationally feasible only with a GPU providing a massive speed boost.

For TtC team purposes, we have found ANN using a compute instance with a lot of RAM and a large number of cores to be the most performant evaluation option. ANN is roughly twice as fast as Exact search, even with GPU-boosting, due to the speed of retrieval. GPU-boosted exact search is in turn ~10 times faster than exact search without GPU-boosting. For most use-cases, we advise using ANN.

In [5]:
import torch

USE_EXACT_SEARCH = False

if USE_EXACT_SEARCH:
    assert torch.cuda.is_available()

## Step 1: Create File Mount

The Azure Machine Learning File Mount system, though cumbersomely named, allows us to _directly_ access files and objects we have stored in the DIBBs TTC container. Any `.txt` or `.csv` files need to be created as Data Assets (see sidebar on left), while embedding tensor files and any HNSW `.index` files do not, and can simply be loaded directly from storage.

In [6]:
# Load up the validation set data
from azureml.fsspec import AzureMachineLearningFileSystem

# Instantiate a file system over the workspace so we can interact with data
# assets directly--we get all the goodies like open, ls, etc.
fs = AzureMachineLearningFileSystem(
    f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}"
)


/anaconda/envs/dibbs_env/lib/python3.10/site-packages/azureml/dataprep/api/_loggerfactory.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Step 2: Concatenate JSONL Embeddings

Using our mounted file system, we can directly open the embedding file file and load all JSONLs present in it. Remember, each embedding file is stored as a dictionary of not just the embeddings computed by the `sentence-transformers` model, but the standard LOINC codes associated with those embeddings. These are important later for measuring accuracy.

Caveat: On a smaller Azure server, this can take about ~35 min to concatenate and load.

In [ ]:
import json
import numpy as np
import os

EMBEDDING_FILE_LIST = [
    os.path.join(MODEL_FOLDER, os.path.basename(p))
    for p in fs.ls(MODEL_FOLDER)
    if p.endswith(".jsonl")
]

cache_data = []
for file in EMBEDDING_FILE_LIST:
    with fs.open(file) as f:
        cache_data.extend(
            json.loads(line)
            for line in f
            if line.strip()
        )

name_codes = [item["description"] for item in cache_data]
embeddings = [np.array(item["descriptionVector"], dtype=np.float32)
              for item in cache_data]
loinc_type = [item["type"] for item in cache_data]

if not USE_EXACT_SEARCH:
    # embeddings: list of 1D vectors -> stack to 2D array for hnswlib
    embeddings = np.stack(embeddings, axis=0)  # shape (N, D)

## Step 3: Load HNSW Index

Whether computed from a previous Azure run, or computed locally and uploaded, we will use the embeddings to populate an HNSW index for fast Approximate Nearest Neighbor searching. The parameter values below govern the depth / connectivity of the search, but note that if the index was previously constructed, only the `EF_SEARCH` value will impact performance.

The `hnswlib` package _cannot_ directly open Azure binary files, which is how the FileSystemMount accesses and passes them. So what we need to do instead is first copy the file from the remote mount to local, working memory, and then we can access and open it. Once we've done that, it should be locally persisted for the remainder of our session.

During operation, this cell will create a temporary copy of the `.index` file in local, working memory. At the end of the cell, the file will be remote copied to Blob Storage and then deleted from local memory (on subsequent runs, it will simply be fetched from remote storage). You can verify the file has been cleaned by checking the sidebar to the left, under `Notebooks`.


In [12]:
import hnswlib

# ANN INDEX VARIABLES
EF_CONSTRUCTION = 400
M_VALUE = 64
EF_SEARCH = 400

if not USE_EXACT_SEARCH:
    # Load up or create an index over the embedding data
    index = hnswlib.Index(space="cosine", dim=EMBEDDING_SIZE)

    # Azure will check blob storage first using the file mount
    print("Checking for cached ANN index...")
    if fs.exists("indexes/" + INDEX_FP):
        print("  Found cached index. Loading it...")

        # First, try to regularly load the index, in case we copied it here
        # from a previous run
        try:
            index.load_index(INDEX_FP)
        
        # If we can't open the file (because it's AzureML binary), then we
        # can create a local ported copy and open from that
        except:
            try:
                fs.get("indexes/" + INDEX_FP, '.')
                index.load_index(INDEX_FP)
            
            # If that doesn't work then the file is beyond the reach of mortal
            # hands and is best left undisturbed, like all sleeping gods
            except:
                print("Could not copy or load index")
        
    else:
        print("No locally cached index found. Creating hierarchical index...")
        index.init_index(
            max_elements=len(embeddings), ef_construction=EF_CONSTRUCTION, M=M_VALUE
        )
        index.add_items(embeddings, list(range(len(embeddings))))

        # Default is to save to local, working memory, so we'll need to remote copy
        # to Azure blob storage just like the reverse of copying from blob storage
        # Also clean up the local copy to avoid surplus memory charges
        index.save_index(INDEX_FP)
        fs.put(INDEX_FP, "/indexes/")
    os.remove(INDEX_FP)

    # The index should be holding approximately 276k embeddings so it better exceed 0
    assert index.get_current_count() > 0
    index.set_ef(EF_SEARCH)


Checking for cached ANN index...
  Found cached index. Loading it...


## Step 4: Load Validation Set

With our file system mount, loading the validation set and preparing it for evaluation is straightforward. No need to worry about local copying for this data, Azure's `fs.open()` can simply parse the binary into a string codec for us.

In [13]:
print("Loading validation set...")
examples = []
with fs.open(VALIDATION_FILE) as fp:
    for line in fp:
        # Blob storage is bytes-based, so we need to decode before string operations
        line_str = line.decode("utf-8")
        if line_str.strip() != "":
            examples.append(line_str.strip().split("|"))

# There are either 60k examples or 240k examples in this list, depending
# on whether the abridged set or full validation set is used
assert len(examples) >= 60000

Loading validation set...


## Step 5: Perform Evaluation

This cell carries out the trial run with the model and scores its performance on the validation data. It's largely a dictionary-based tracking function that accumulates some numbers into lists partitioned out by the K-value associated with the run. The search method of retrieving results is slightly different depending on whether exact search or ANN is used (i.e. slightly different unpacking of the `hits` list). 

When we use the `hnswlib` API to perform ANN, we get a pretty nested structure of a pair of lists denoting the search results and the _distances_ of those results to the input query. The only nuance to this function is unpacking those lists, converting distances into scores (since we want to measure similarity), and pairing up the found neighbor result with the standard LOINC code it represents, using our earlier unpickled Corpus ID indices.

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers import util

import json
import time
import random

print("Instantiating language model...")
model = SentenceTransformer(MODEL_NAME)

print("Predicting and computing stats for validation set...")

random.shuffle(examples)

JSONL_FP = f"eval_results_{MODEL_NAME.replace('/', '_')}_{DATE}.jsonl"

stats = {
    "encoding_time_sum": 0.0,
    "encoding_time_n": 0,
    "by_k": {
        str(k): {
            "search_time_sum": 0.0,
            "search_time_n": 0,
            "highest_cos_sim_sum": 0.0,
            "highest_cos_sim_n": 0,
            "correct_in_topk": 0,
            "examples_n": 0,
            "rank_sum_when_present": 0.0,
            "rank_n_when_present": 0,
            "right_cos_sim_sum_when_present": 0.0,
            "right_cos_sim_n_when_present": 0,
        }
        for k in K_VALUES
    },
}


with open(JSONL_FP, "w", encoding="utf-8") as fp:
    for i, e in enumerate(examples):
        if i % 10_000 == 0:
            print(f"Calculated {i} of {len(examples)} examples.")

        expected_label = e[0].strip()
        query_input = e[1].strip()

        enc_start = time.time()
        if USE_EXACT_SEARCH:
            enc = model.encode(query_input, convert_to_tensor=True)
        else:
            enc = model.encode(query_input)
        enc_time = time.time() - enc_start

        stats["encoding_time_sum"] += float(enc_time)
        stats["encoding_time_n"] += 1

        for k in K_VALUES:
            k_str = str(k)
            stats["by_k"][k_str]["examples_n"] += 1

            search_start = time.time()
            if USE_EXACT_SEARCH:
                raw_hits = util.semantic_search(enc, embeddings, top_k=k)[0]
                hits = [
                    {"corpus_id": int(h["corpus_id"]), "score": float(h["score"])}
                    for h in raw_hits
                ]
            else:
                embedding_ids, distances = index.knn_query(enc, k=k)
                hits = [
                    {"corpus_id": int(corpus_id), "score": float(1 - dist)}
                    for corpus_id, dist in zip(embedding_ids[0], distances[0])
                ]
                hits = sorted(hits, key=lambda x: x["score"], reverse=True)
            search_time = time.time() - search_start

            stats["by_k"][k_str]["search_time_sum"] += float(search_time)
            stats["by_k"][k_str]["search_time_n"] += 1

            top_score = float(hits[0]["score"])
            stats["by_k"][k_str]["highest_cos_sim_sum"] += top_score
            stats["by_k"][k_str]["highest_cos_sim_n"] += 1

            top_corpus_id = int(hits[0]["corpus_id"])
            top_label = name_codes[top_corpus_id]
            top_loinc_type = loinc_type[top_corpus_id]
            is_correct_top1 = top_label == expected_label

            expected_rank = None
            expected_score = None

            for idx, h in enumerate(hits):
                candidate_label = name_codes[int(h["corpus_id"])]
                if candidate_label == expected_label:
                    expected_rank = idx + 1
                    expected_score = float(h["score"])
                    break

            is_correct_in_topk = expected_rank is not None

            if is_correct_in_topk:
                stats["by_k"][k_str]["correct_in_topk"] += 1
                stats["by_k"][k_str]["rank_sum_when_present"] += float(expected_rank)
                stats["by_k"][k_str]["rank_n_when_present"] += 1
                stats["by_k"][k_str]["right_cos_sim_sum_when_present"] += float(expected_score)
                stats["by_k"][k_str]["right_cos_sim_n_when_present"] += 1

            row = {
                "example_idx": i,
                "query_input": query_input,
                "expected_label": expected_label,
                "k": k,
                "encoding_time_s": float(enc_time),
                "search_time_s": float(search_time),
                "expected_match": {
                    "rank": expected_rank,
                    "score": expected_score,
                    "is_correct_in_topk": is_correct_in_topk,
                    "is_correct_top1": is_correct_top1,
                },
                "results": [
                    {
                        "rank": idx + 1,
                        "corpus_id": int(h["corpus_id"]),
                        "label": name_codes[int(h["corpus_id"])],
                        "loinc_type": loinc_type[int(h["corpus_id"])],
                        "score": float(h["score"]),
                    }
                    for idx, h in enumerate(hits)
                ],
            }

            fp.write(json.dumps(row, ensure_ascii=False) + "\n")

mean_encoding_time = round(
    float(stats["encoding_time_sum"]) / float(stats["encoding_time_n"]),
    3,
)
print(f"  Mean Encoding Time: {mean_encoding_time} seconds")

for k in K_VALUES:
    k_str = str(k)
    ex_n = float(stats["by_k"][k_str]["examples_n"])

    top_k_accuracy = round(float(stats["by_k"][k_str]["correct_in_topk"]) / ex_n, 5)

    mean_high_cosine_sim = round(
        float(stats["by_k"][k_str]["highest_cos_sim_sum"]) / float(stats["by_k"][k_str]["highest_cos_sim_n"]),
        3,
    )

    mean_search_time = round(
        float(stats["by_k"][k_str]["search_time_sum"]) / float(stats["by_k"][k_str]["search_time_n"]),
        3,
    )

    if stats["by_k"][k_str]["rank_n_when_present"] > 0:
        mean_rank = round(
            float(stats["by_k"][k_str]["rank_sum_when_present"]) / float(stats["by_k"][k_str]["rank_n_when_present"]),
            3,
        )
        mean_right_cos_sim = round(
            float(stats["by_k"][k_str]["right_cos_sim_sum_when_present"]) / float(stats["by_k"][k_str]["right_cos_sim_n_when_present"]),
            3,
        )
    else:
        mean_rank = None
        mean_right_cos_sim = None

    print(f"  Trial: Value for Top-K at K = {k}")
    print(f"    Top-K Accuracy: {top_k_accuracy * 100.0}%")
    print(f"    Mean Rank of Correct Code (when present): {mean_rank}")
    print(f"    Mean Highest Cosine Similarity: {mean_high_cosine_sim}")
    print(f"    Mean Correct Cosine Similarity: {mean_right_cos_sim}")
    print(f"    Mean Search Time: {mean_search_time}")

print("Wrote:")
print(" ", JSONL_FP)



Instantiating language model...
Predicting and computing stats for validation set...
Calculated 0 of 60000 examples.
